In [69]:
from openai import AsyncClient
import os
from pydantic import BaseModel, ConfigDict
from pprint import pprint


In [114]:
async_client = AsyncClient(base_url=os.getenv('ORQ_BASE_URL'), api_key=os.getenv('ORQ_API_KEY'))

class Address(BaseModel):
    model_config = dict(extra="forbid")
    street: str
    city: str

class PersonalInformation(BaseModel):
    model_config = dict(extra="forbid")

    name: str
    subanswer: Address


pprint(PersonalInformation.model_json_schema())


response = await async_client.chat.completions.parse(
    messages=[{'role': 'user', 'content': 'give me some random text'}],
    model='azure/gpt-4.1-mini',
    response_format=PersonalInformation

)
print(response.choices[0].message.parsed)

## Fails
response = await async_client.chat.completions.parse(
    messages=[{'role': 'user', 'content': 'give me some random text'}],
    model='google-ai/gemini-2.5-flash',
    response_format=PersonalInformation

)
print(response.choices[0].message.parsed)


{'$defs': {'Address': {'additionalProperties': False,
                       'properties': {'city': {'title': 'City',
                                               'type': 'string'},
                                      'street': {'title': 'Street',
                                                 'type': 'string'}},
                       'required': ['street', 'city'],
                       'title': 'Address',
                       'type': 'object'}},
 'additionalProperties': False,
 'properties': {'name': {'title': 'Name', 'type': 'string'},
                'subanswer': {'$ref': '#/$defs/Address'}},
 'required': ['name', 'subanswer'],
 'title': 'PersonalInformation',
 'type': 'object'}
name='Random Text' subanswer=Address(street='123 Imaginary Lane', city='Faketown')


BadRequestError: Error code: 400 - {'code': 400, 'error': '{"error":{"code":400,"message":"Invalid JSON payload received. Unknown name \\"$ref\\" at \'generation_config.response_schema.properties[1].value\': Cannot find field.","status":"INVALID_ARGUMENT","details":[{"@type":"type.googleapis.com/google.rpc.BadRequest","fieldViolations":[{"field":"generation_config.response_schema.properties[1].value","description":"Invalid JSON payload received. Unknown name \\"$ref\\" at \'generation_config.response_schema.properties[1].value\': Cannot find field."}]}]}}', 'source': 'provider'}

In [31]:
google_models = [
      "google/gemini-1.0-pro-001",
      "google/gemini-1.0-pro-vision-001",
      "google/gemini-2.0-flash-001",
      "google/gemini-2.0-flash-exp",
      "google/gemini-2.0-flash-lite-001",
      "google/gemini-2.5-flash",
      "google/gemini-2.5-flash-lite-preview-06-17",
      "google/gemini-2.5-flash-preview-04-17",
      "google/gemini-2.5-flash-preview-05-20",
      "google/gemini-2.5-pro",
      "google/meta/llama-3.3-70b-instruct-maas",
      "google/mistral-large-2411",
      "google-ai/gemini-1.0-pro",
      "google-ai/gemini-1.5-flash",
      "google-ai/gemini-1.5-flash-8b-exp-0827",
      "google-ai/gemini-1.5-flash-exp-0827",
      "google-ai/gemini-1.5-pro",
      "google-ai/gemini-2.0-flash",
      "google-ai/gemini-2.0-flash-001",
      "google-ai/gemini-2.0-flash-exp",
      "google-ai/gemini-2.0-flash-lite-001",
      "google-ai/gemini-2.0-flash-thinking-exp-01-21",
      "google-ai/gemini-2.0-pro-exp-02-05",
      "google-ai/gemini-2.5-flash",
      "google-ai/gemini-2.5-flash-lite-preview-06-17",
      "google-ai/gemini-2.5-flash-preview-04-17",
      "google-ai/gemini-2.5-flash-preview-05-20",
      "google-ai/gemini-2.5-pro",
      'google/claude-sonnet-4@20250514',
      'google/claude-3-5-sonnet@20240620',
      'google/meta/llama-3.3-70b-instruct-maas'
  ]

for model in google_models:
    try:
        response = await async_client.chat.completions.parse(
            messages=[{'role': 'user', 'content': 'give me some random text'}],
            model=model,
            response_format=PersonalInformation
        )
        print(f'{model} success')
    except:
        print(f'{model} failed')

google/gemini-1.0-pro-001 failed
google/gemini-1.0-pro-vision-001 failed
google/gemini-2.0-flash-001 failed
google/gemini-2.0-flash-exp failed
google/gemini-2.0-flash-lite-001 failed
google/gemini-2.5-flash failed
google/gemini-2.5-flash-lite-preview-06-17 failed
google/gemini-2.5-flash-preview-04-17 failed
google/gemini-2.5-flash-preview-05-20 failed
google/gemini-2.5-pro failed
google/meta/llama-3.3-70b-instruct-maas failed
google/mistral-large-2411 failed
google-ai/gemini-1.0-pro failed
google-ai/gemini-1.5-flash failed
google-ai/gemini-1.5-flash-8b-exp-0827 failed
google-ai/gemini-1.5-flash-exp-0827 failed
google-ai/gemini-1.5-pro failed
google-ai/gemini-2.0-flash failed
google-ai/gemini-2.0-flash-001 failed
google-ai/gemini-2.0-flash-exp failed
google-ai/gemini-2.0-flash-lite-001 failed
google-ai/gemini-2.0-flash-thinking-exp-01-21 failed
google-ai/gemini-2.0-pro-exp-02-05 failed
google-ai/gemini-2.5-flash failed
google-ai/gemini-2.5-flash-lite-preview-06-17 failed
google-ai/gemin

In [128]:
import jsonref
replaced_refs = jsonref.replace_refs(PersonalInformation.model_json_schema(), lazy_load=False, proxies=False)
replaced_refs


{'$defs': {'Address': {'additionalProperties': False,
   'properties': {'street': {'title': 'Street', 'type': 'string'},
    'city': {'title': 'City', 'type': 'string'}},
   'required': ['street', 'city'],
   'title': 'Address',
   'type': 'object'}},
 'additionalProperties': False,
 'properties': {'name': {'title': 'Name', 'type': 'string'},
  'subanswer': {'additionalProperties': False,
   'properties': {'street': {'title': 'Street', 'type': 'string'},
    'city': {'title': 'City', 'type': 'string'}},
   'required': ['street', 'city'],
   'title': 'Address',
   'type': 'object'}},
 'required': ['name', 'subanswer'],
 'title': 'PersonalInformation',
 'type': 'object'}

In [170]:
from openai.types.shared_params.response_format_json_schema import ResponseFormatJSONSchema, JSONSchema

json_schema = JSONSchema(
    name=replaced_refs['title'],  # Required field
    schema=replaced_refs,     # Your actual schema dict
    strict=False,              # Optional
    # description=replaced_refs['title']
)

response = await async_client.chat.completions.create(
    messages=[{'role': 'user', 'content': 'give me some random text'}],
    model='google-ai/gemini-2.5-flash',
    # response_format=schema
    response_format=ResponseFormatJSONSchema(type="json_schema", json_schema=json_schema),
)
print(response.choices[0].message.content)

{"random_text": "The quick brown fox jumps over the lazy dog."}


In [ ]:
from openai import OpenAI

client = OpenAI(
    api_key="REDACTED_GOOGLE_KEY",
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

## Succeeds with google's openai endpoint
client.chat.completions.parse(
    messages=[{'role': 'user', 'content': 'give me some random text'}],
    model='gemini-2.5-flash',
    response_format=PersonalInformation
).choices[0].message.parsed

NotFoundError: Error code: 404 - [{'error': {'code': 404, 'message': 'models/google/gemini-2.5-flash is not found for API version v1main, or is not supported for generateContent. Call ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}]

In [134]:

for model in google_models:
    try:
        model_str = model.split('/')[1]
        response = client.chat.completions.parse(
            messages=[{'role': 'user', 'content': 'give me some random text'}],
            model=model_str,
            response_format=PersonalInformation
        )
        print(f'{model} success')
    except:
        print(f'{model} failed')

google/gemini-1.0-pro-001 failed
google/gemini-1.0-pro-vision-001 failed
google/gemini-2.0-flash-001 success
google/gemini-2.0-flash-exp success
google/gemini-2.0-flash-lite-001 success
google/gemini-2.5-flash success
google/gemini-2.5-flash-lite-preview-06-17 success
google/gemini-2.5-flash-preview-04-17 failed
google/gemini-2.5-flash-preview-05-20 success
google/gemini-2.5-pro success
google/meta/llama-3.3-70b-instruct-maas failed
google/mistral-large-2411 failed
google-ai/gemini-1.0-pro failed
google-ai/gemini-1.5-flash success
google-ai/gemini-1.5-flash-8b-exp-0827 failed
google-ai/gemini-1.5-flash-exp-0827 failed
google-ai/gemini-1.5-pro failed
google-ai/gemini-2.0-flash success
google-ai/gemini-2.0-flash-001 success
google-ai/gemini-2.0-flash-exp success
google-ai/gemini-2.0-flash-lite-001 success
google-ai/gemini-2.0-flash-thinking-exp-01-21 success
google-ai/gemini-2.0-pro-exp-02-05 failed
google-ai/gemini-2.5-flash success
google-ai/gemini-2.5-flash-lite-preview-06-17 success


In [136]:
os.getenv('ORQ_BASE_URL')

'https://api.orq.ai/v2/proxy'

In [186]:
from openai import Client
async_client = AsyncClient(base_url='http://localhost:3800/v2/proxy', api_key='REDACTED_ORQ_KEY')
client = Client(
    base_url='http://localhost:3800/v2/proxy',
    api_key='REDACTED_ORQ_KEY'
)

class Address(BaseModel):
    model_config = dict(extra="forbid")
    street: str
    city: str

class PersonalInformation(BaseModel):
    model_config = dict(extra="forbid")

    name: str
    subanswer: Address


pprint(PersonalInformation.model_json_schema())


# response = await async_client.chat.completions.parse(
#     messages=[{'role': 'user', 'content': 'give me some random text'}],
#     model='azure/gpt-4.1-mini',
#     response_format=PersonalInformation

# )
# print(response.choices[0].message.parsed)

## Fails
response = await async_client.chat.completions.parse(
    messages=[{'role': 'user', 'content': 'give me some random text'}],
    model='google-ai/gemini-2.5-flash',
    response_format=PersonalInformation

)
response.choices[0].message.parsed

{'$defs': {'Address': {'additionalProperties': False,
                       'properties': {'city': {'title': 'City',
                                               'type': 'string'},
                                      'street': {'title': 'Street',
                                                 'type': 'string'}},
                       'required': ['street', 'city'],
                       'title': 'Address',
                       'type': 'object'}},
 'additionalProperties': False,
 'properties': {'name': {'title': 'Name', 'type': 'string'},
                'subanswer': {'$ref': '#/$defs/Address'}},
 'required': ['name', 'subanswer'],
 'title': 'PersonalInformation',
 'type': 'object'}


PersonalInformation(name='RandomUser', subanswer=Address(street='123 Main St', city='Anytown'))

In [179]:
response = await async_client.responses.parse(
    input='fill this with fake data',
    model='google-ai/gemini-2.5-flash',
    text_format=PersonalInformation
)


InternalServerError: Error code: 500 - {'code': 500, 'error': 'createResponseNonStreaming not implemented by provider', 'source': 'system'}

In [172]:
response = await async_client.chat.completions.parse(
    messages=[{'role': 'user', 'content': 'give me some random text'}],
    model='google-ai/gemini-2.0-flash',
    response_format=PersonalInformation,
)
response.choices[0].message.parsed

ValidationError: 1 validation error for PersonalInformation
  Input should be an object [type=model_type, input_value=['The quick brown fox jum... that is the question.'], input_type=list]
    For further information visit https://errors.pydantic.dev/2.11/v/model_type

In [166]:
response = await async_client.chat.completions.create(
    messages=[{'role': 'user', 'content': 'give me some random text'}],
    model='google-ai/gemini-2.5-flash',
    response_format=ResponseFormatJSONSchema(type="json_schema", json_schema=json_schema),
)
print(response.choices[0].message.content)

{"name":"Random Name","subanswer":{"street":"123 Main St","city":"Anytown"}}


In [169]:
PersonalInformation.model_json_schema()

{'$defs': {'Address': {'additionalProperties': False,
   'properties': {'street': {'title': 'Street', 'type': 'string'},
    'city': {'title': 'City', 'type': 'string'}},
   'required': ['street', 'city'],
   'title': 'Address',
   'type': 'object'}},
 'additionalProperties': False,
 'properties': {'name': {'title': 'Name', 'type': 'string'},
  'subanswer': {'$ref': '#/$defs/Address'}},
 'required': ['name', 'subanswer'],
 'title': 'PersonalInformation',
 'type': 'object'}